# Template 06: SHAP DataFrame Creation

**Purpose:** Compute SHAP values and create SHAP dataframes

**Inputs:**
- models/xgb_model.json
- data/04c_train_encoded.parquet
- data/04c_test_encoded.parquet

**Outputs:**
- results/06_shap_train.parquet
- results/06_shap_test.parquet

In [1]:
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_coll/v1"


In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import yaml
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 06: SHAP DATAFRAME CREATION")
print("########################################")

project_root = setup_notebook_environment()

########################################
# STAGE 06: SHAP DATAFRAME CREATION
########################################


In [4]:
print(f"Python: {sys.version}")
print(f"XGBoost: {xgb.__version__}")
print(f"SHAP: {shap.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print("=" * 50)
print()

Python: 3.11.15 (main, Jun 11 2026, 15:14:57) [Clang 20.1.8 ]
XGBoost: 3.1.1
SHAP: 0.51.0
Pandas: 3.0.5
NumPy: 2.4.6



In [5]:
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

output_base = cfg['paths']['output_base']
target = cfg['experiment']['target']

In [6]:
# Load model (use Booster for XGBoost 2.0+ compatibility)
model_file = f"{output_base}/models/xgb_model.json"
booster = xgb.Booster()
booster.load_model(model_file)
print(f"\n* Model loaded: {model_file}")


* Model loaded: output/car_coll/v1/models/xgb_model.json


In [7]:
# Load ENCODED data from Stage 04c
X_train = pd.read_parquet(f"{output_base}/data/04c_train_encoded.parquet")
X_test = pd.read_parquet(f"{output_base}/data/04c_test_encoded.parquet")

print(f"\n* Train encoded: {X_train.shape}")
print(f"* Test encoded: {X_test.shape}")


* Train encoded: (3757142, 197)
* Test encoded: (3750266, 197)


In [8]:
# Create SHAP explainer (using booster for XGBoost 2.0+ compatibility)
print(f"\n* Creating SHAP explainer...")
explainer = shap.TreeExplainer(booster)
print(f"  Explainer created")


* Creating SHAP explainer...
  Explainer created


In [9]:
# Compute SHAP values for train (sample if too large)
print(f"\n* Computing SHAP values for train...")
sample_size = min(10000, len(X_train))
X_train_sample = X_train.sample(n=sample_size, random_state=42)

shap_values_train = explainer.shap_values(X_train_sample)
print(f"  SHAP values shape: {shap_values_train.shape}")

# Create SHAP dataframe
shap_df_train = pd.DataFrame(shap_values_train, columns=X_train.columns, index=X_train_sample.index)
shap_df_train['base_value'] = explainer.expected_value
print(f"  SHAP dataframe created: {shap_df_train.shape}")


* Computing SHAP values for train...


  SHAP values shape: (10000, 197)
  SHAP dataframe created: (10000, 198)


In [10]:
# Compute SHAP values for test
print(f"\n* Computing SHAP values for test...")
shap_values_test = explainer.shap_values(X_test)
print(f"  SHAP values shape: {shap_values_test.shape}")

# Create SHAP dataframe
shap_df_test = pd.DataFrame(shap_values_test, columns=X_train.columns, index=X_test.index)
shap_df_test['base_value'] = explainer.expected_value
print(f"  SHAP dataframe created: {shap_df_test.shape}")


* Computing SHAP values for test...


  SHAP values shape: (3750266, 197)


  SHAP dataframe created: (3750266, 198)


In [11]:
# Save SHAP dataframes
shap_train_file = f"{output_base}/results/06_shap_train.parquet"
shap_test_file = f"{output_base}/results/06_shap_test.parquet"

shap_df_train.to_parquet(shap_train_file)
shap_df_test.to_parquet(shap_test_file)

print(f"\n* Saved:")
print(f"  {shap_train_file}")
print(f"  {shap_test_file}")


* Saved:
  output/car_coll/v1/results/06_shap_train.parquet
  output/car_coll/v1/results/06_shap_test.parquet


In [12]:
# Feature importance (mean absolute SHAP)
feature_importance = pd.DataFrame({
    'feature': X_train.columns.tolist(),
    'mean_abs_shap': np.abs(shap_values_test).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

importance_file = f"{output_base}/results/06_feature_importance.csv"
feature_importance.to_csv(importance_file, index=False)

print(f"\n* Feature importance saved: {importance_file}")
print(f"\nTop 10 features:")
print(feature_importance.head(10))


* Feature importance saved: output/car_coll/v1/results/06_feature_importance.csv

Top 10 features:
                                feature  mean_abs_shap
4                 multi_pol_unknown_cal       0.298440
93                      vc_camera_raw_0       0.295280
164                 vc_seating_rows_raw       0.160352
6                           married_ind       0.139403
112  vc_electronic_braking_system_raw_5       0.134356
47             cef_est_curr_mi_grp_imps       0.133368
52                  cef_lien_holder_ind       0.131592
95                      vc_camera_raw_5       0.113317
50              cef_first_poten_dam_ind       0.100482
3                     multi_pol_yes_cal       0.095246


In [13]:
print("\n########################################")
print("# STAGE 06: COMPLETE")
print("########################################")


########################################
# STAGE 06: COMPLETE
########################################
